In [4]:
import os, sys, pathlib, importlib
import pandas as pd
import json
import time
from tqdm import tqdm
sys.path.append(os.path.abspath(".."))
from utils import preprocess_images
from utils.preprocess_images import download_images_batch


## clf

In [34]:
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
from ultralytics import YOLO

# ----- 1. MODELS -----
# YOLOv8 人物检测
yolo_model = YOLO("yolov8n.pt")  # 或 yolov8n-seg 根据需要，n是轻量模型

# CLIP 文本 + 图像
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

## 标签文本
labels = [
    "Professional portrait: ONLY one person, front-facing, occupying most of the image, no meaningful background",
    "Lifestyle photo: person in everyday life or outdoor context, engaged in activities or travel, visible and richer background"
    ]

# ----- 2. DATA -----
# IMG_FOLDER="images\host_pic_sample"
IMG_FOLDER="host_pic_paris_43258"
images=[]

## marche le mieux quand on fournit sys et user prompt à la fois
start_time=time.time()

for filename in os.listdir(IMG_FOLDER):
    if not filename.endswith((".jpg",".JPG",".jpeg",".png","tif")):
        continue #==skip
    img_path=os.path.join(IMG_FOLDER, filename)
    images.append(img_path)
print(f"{len(images)} images")


# ----- 3. Pipeline -----
results = []
SAVE_EVERY=1000
for i, img_path in enumerate(tqdm(images[:1001], desc="classify host profile picture...")):
    image = Image.open(img_path).convert("RGB")
    
    # # YOLO 人物检测
    yolo_preds = yolo_model(source=image, verbose=False)

    # yolo_preds = yolo_model(image,conf=0.12)#调整置信度
    if len(yolo_preds[0].boxes) == 0:
        results.append({
            "image": img_path,
            "host_id":os.path.splitext(os.path.basename(img_path))[0],
            "label": "no_person",
            "confidence": 1.0,
            "bbox": None
        })
        continue

    # CLIP 分类同之前代码
    inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True)
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)
    conf, idx = probs.max(dim=1)
    pred_label = ["pro_style", "life_style"][idx.item()]  # 对应原始分类标签
    
    results.append({
        "image": img_path,
        "host_id":os.path.splitext(os.path.basename(img_path))[0],
        # "label": labels[idx.item()],
        "label":pred_label,
        "confidence": conf.item(),
        "bbox": [box.xyxy.tolist() for box in yolo_preds[0].boxes]  # YOLO检测框
    })
    if i!=0 and i % SAVE_EVERY == 0:# //整除，%取余数
        path_result="host_pic_paris_43258/results_clf.json"
        with open(path_result,'w', encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)            
            print(f"save intermediate results: first {i} / {len(images)} images")
        
end_time=time.time()
print(f"\n [DONE] classify {len(results)} images: {end_time-start_time:.2f} sec! \n")

path_result="host_pic_paris_43258/results_clf.json"
with open(path_result,'w', encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"[SAVE] {len(results)} results saved to {path_result}!\n")
    

43055 images


classify host profile picture...: 100%|██████████| 1001/1001 [02:40<00:00,  6.24it/s]

save intermediate results: first 1000 / 43055 images

 [DONE] classify 1001 images: 160.67 sec! 

[SAVE] 1001 results saved to host_pic_paris_43258/results_clf.json!



In [36]:
## copie-coller
import os
import shutil
from pathlib import Path

# 原始图片根目录
# IMG_FOLDER="host_pic_paris_43258"

with open(path_result,"r", encoding="utf-8")as f:
    results=json.load(f)


# 人工检查目录
IMG_FOLDER_SAMPLE = Path(r"host_pic_sample1000")
os.makedirs(IMG_FOLDER_SAMPLE, exist_ok=True)
# IMG_FOLDER_SAMPLE.mkdir(parents=True, exist_ok=True)

start_time=time.time()

for i, item in enumerate(tqdm(results, desc=f"shutil pic to {IMG_FOLDER_SAMPLE}...")):
    # 统一路径分隔符（防止 \ 和 / 混用）
    src_path = Path(item['image'])
    filename=item["host_id"]
    type=item["label"]
    
    dst_path=os.path.join(IMG_FOLDER_SAMPLE, type,  f"{filename}.jpg")
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    
    # shutil
    if src_path.exists():
        shutil.copy2(src_path, dst_path)
    else:
        print("找不到:", src_path)
end_time=time.time()

print(f"[DONE] {i+1} images shutiled in {end_time-start_time:.2f} sec!")


shutil pic to host_pic_sample1000...: 100%|██████████| 1001/1001 [00:01<00:00, 844.90it/s]

[DONE] 1001 images shutiled in 1.19 sec!


In [30]:
dst_path

'host_pic_sample1000\\life_style\\10014012.jpg'

In [22]:
## check results
with open(path_result,"r", encoding="utf-8")as f:
    results=json.load(f)
print(len(results))
print(results[0])

1001
{'image': 'host_pic_paris_43258\\100004988.jpg', 'host_id': '100004988', 'label': 'pro_style', 'confidence': 0.9665592312812805, 'bbox': [[[12.649340629577637, 21.16490364074707, 207.73704528808594, 224.56495666503906]]]}


In [ ]:
# path_result="images\host_pic_sample/results_clf3.json"

import json
with open("images\host_pic_sample/metadata_host_pic.json","r", encoding="utf-8")as f:
    metadata=json.load(f)
with open(path_result,"r", encoding="utf-8")as f:
    results=json.load(f)

import pandas as pd

host_data=[]
for  host in metadata:
    host_id=host["host_id"]
    # print(type(host_id))
    pred_label=[res['label'] for res in results if res['host_id']==str(host_id)][0]
    true_label=host["true_label"]
    host_data.append({"host_id":host_id,
                      'true_label':true_label,
                      'pred_label':pred_label}
                     )
    # print(f"host_id:{host_id}, true label:{true_label}, pred label:{pred_label} \n")
    # break
host_data_df=pd.DataFrame(host_data)
display(host_data_df)

host_data_df['tp']=host_data_df.apply(lambda row : 1 if str(row["true_label"])==str(row["pred_label"]) else 0, axis=1)
print(host_data_df['tp'].mean())

,host_id,true_label,pred_label
0,102571900,pro_style,pro_style
1,106294215,life_style,no_person
2,106365215,life_style,life_style
3,137154154,life_style,life_style
4,212791574,no_person,no_person
5,2379345,no_person,no_person
6,24654560,life_style,life_style
7,2798386,pro_style,pro_style
8,28470251,pro_style,pro_style
9,32741638,pro_style,no_person


0.8


## download pics

In [2]:
import pandas as pd
df_all=pd.read_csv("../data_all\listings_paris_london2406.csv")
print(df_all.shape)

(112594, 129)


C:\Users\yeliu\AppData\Local\Temp\ipykernel_21216\4751896.py:2: DtypeWarning: Columns (68,77,79) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all=pd.read_csv("../data_all\listings_paris_london2406.csv")


In [3]:
df_pic=df_all.drop_duplicates(subset='host_picture_url')
df_pic_paris=df_pic[df_pic['city']=="paris"]

print(df_pic.shape, df_pic_paris.shape)
df_pic_paris[['host_id','host_picture_url']].value_counts(dropna=False, ascending=False)

(68309, 129) (43258, 129)


host_id    host_picture_url                                                                                                                      
582456761  https://a0.muscache.com/im/pictures/user/User-582456761/original/14ac25b8-1ecb-4eef-9180-7771bb120875.jpeg?aki_policy=profile_x_medium    1
2626       https://a0.muscache.com/im/pictures/user/ad6a9447-d6fa-4b6b-a820-1c5b52cd5359.jpg?aki_policy=profile_x_medium                             1
3631       https://a0.muscache.com/im/users/3631/profile_pic/1375800198/original.jpg?aki_policy=profile_x_medium                                     1
5468       https://a0.muscache.com/im/pictures/user/User-5468/original/96c3d338-c643-4424-8bec-af91deeedf20.jpeg?aki_policy=profile_x_medium         1
6792       https://a0.muscache.com/im/pictures/user/e079f529-45b3-4fdb-9066-f63316ab2407.jpg?aki_policy=profile_x_medium                             1
                                                                                                   

In [4]:
df_pic_paris_sample=df_pic_paris[:10]
# display(df_pic_paris_sample)
# id_url_list: list of tuples [(host_id, url), ...]
id_url_list=[(row['host_id'], row["host_picture_url"]) for _, row in df_pic_paris_sample.iterrows()]
print(id_url_list)

[(3631, 'https://a0.muscache.com/im/users/3631/profile_pic/1375800198/original.jpg?aki_policy=profile_x_medium'), (433758, 'https://a0.muscache.com/im/users/433758/profile_pic/1330955021/original.jpg?aki_policy=profile_x_medium'), (7903, 'https://a0.muscache.com/im/users/7903/profile_pic/1280002723/original.jpg?aki_policy=profile_x_medium'), (2626, 'https://a0.muscache.com/im/pictures/user/ad6a9447-d6fa-4b6b-a820-1c5b52cd5359.jpg?aki_policy=profile_x_medium'), (228508, 'https://a0.muscache.com/im/users/228508/profile_pic/1284123543/original.jpg?aki_policy=profile_x_medium'), (22155, 'https://a0.muscache.com/im/pictures/user/975c3587-7e98-490e-a35e-b39ed09b8355.jpg?aki_policy=profile_x_medium'), (28422, 'https://a0.muscache.com/im/users/28422/profile_pic/1319547089/original.jpg?aki_policy=profile_x_medium'), (33534, 'https://a0.muscache.com/im/pictures/user/4f775bb4-9080-4943-aa57-d516708aec81.jpg?aki_policy=profile_x_medium'), (37107, 'https://a0.muscache.com/im/users/37107/profile_pic

In [5]:
importlib.reload(preprocess_images)
from utils.preprocess_images import download_images_batch
download_images_batch(df_pic_paris, out_dir='host_pic_paris_43258', max_workers=12)


len df:43258!


[WARNING] 无效 URL: https://a0.muscache.com/im/users/295001/profile_pic/1371196555/original.jpg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/4693c6a2-64c1-44df-9cdf-0b6e93507f7f.jpg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/users/5947575/profile_pic/1366192247/original.jpg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/users/7327717/profile_pic/1373194686/original.jpg?aki_policy=profile_x_medium


...

[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/User-523953429/original/6245003f-1f44-49c3-b7d7-6587e40b28b2.jpeg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/User-575732047/original/7a107cae-1fea-40c8-bdb8-6c67a730d43c.jpeg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/users/9515983/profile_pic/1383777330/original.jpg?aki_policy=profile_x_medium



[SUCCES] downloaded 43258 pics : 7327.01 sec!

success: 43052 pics!
fails: 206 pics!


{22155: 'host_pic_paris_43258\\22155.jpg',
 439130: 'host_pic_paris_43258\\439130.jpg',
 228508: 'host_pic_paris_43258\\228508.jpg',
 7903: 'host_pic_paris_43258\\7903.jpg',
 433758: 'host_pic_paris_43258\\433758.jpg',
 3631: 'host_pic_paris_43258\\3631.jpg',
 37107: 'host_pic_paris_43258\\37107.jpg',
 28422: 'host_pic_paris_43258\\28422.jpg',
 429406: 'host_pic_paris_43258\\429406.jpg',
 41718: 'host_pic_paris_43258\\41718.jpg',
 2626: 'host_pic_paris_43258\\2626.jpg',
 33534: 'host_pic_paris_43258\\33534.jpg',
 64627: 'host_pic_paris_43258\\64627.jpg',
 48733: 'host_pic_paris_43258\\48733.jpg',
 44444: 'host_pic_paris_43258\\44444.jpg',
 152242: 'host_pic_paris_43258\\152242.jpg',
 296615: 'host_pic_paris_43258\\296615.jpg',
 64055: 'host_pic_paris_43258\\64055.jpg',
 79843: 'host_pic_paris_43258\\79843.jpg',
 72713: 'host_pic_paris_43258\\72713.jpg',
 464019: 'host_pic_paris_43258\\464019.jpg',
 69389: 'host_pic_paris_43258\\69389.jpg',
 276063: 'host_pic_paris_43258\\276063.jpg',
 

NameError: name 'results' is not defined